# Synthetic Data Validation

Validates all 5 synthetic setups: censoring rates, KM curves, covariate distributions, and setup-specific checks.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import KaplanMeierFitter, CoxPHFitter

from src.data.generate import (
    generate_setup_1, generate_setup_2, generate_setup_3,
    generate_setup_4, generate_setup_5,
)

sns.set_style('whitegrid')
%matplotlib inline

## Setup 1 — Clean PH
Expect: ~30-40% censoring, Cox PH recovers true betas, Schoenfeld residuals do NOT reject PH.

In [ ]:
data1 = generate_setup_1()
print(data1.summary())

In [ ]:
# Kaplan-Meier curve
kmf = KaplanMeierFitter()
kmf.fit(data1.T, event_observed=data1.E)

fig, ax = plt.subplots(figsize=(8, 5))
kmf.plot_survival_function(ax=ax)
ax.set_title('Setup 1 — Kaplan-Meier Survival Curve')
ax.set_xlabel('Time')
ax.set_ylabel('S(t)')
plt.tight_layout()
plt.show()

In [ ]:
# Fit Cox PH and compare recovered betas to ground truth
df1 = data1.to_dataframe()
cph = CoxPHFitter()
cph.fit(df1, duration_col='T', event_col='E')

print('True betas vs recovered:')
for i, name in enumerate(data1.feature_names):
    true_b = data1.true_betas[i]
    est_b = cph.params_[name]
    print(f'  {name:15s}  true={true_b:+.3f}  est={est_b:+.3f}')

cph.print_summary()

In [ ]:
# Schoenfeld residuals test — should NOT reject PH
cph.check_assumptions(df1, p_value_threshold=0.05, show_plots=True)

## Setup 2 — Non-PH
Expect: ~40-50% censoring, Schoenfeld residuals REJECT PH for X_cont_1.

In [ ]:
data2 = generate_setup_2()
print(data2.summary())

In [ ]:
kmf2 = KaplanMeierFitter()
kmf2.fit(data2.T, event_observed=data2.E)

fig, ax = plt.subplots(figsize=(8, 5))
kmf2.plot_survival_function(ax=ax)
ax.set_title('Setup 2 — Kaplan-Meier Survival Curve')
plt.tight_layout()
plt.show()

In [ ]:
# Fit Cox PH — should show PH violation for X_cont_1
df2 = data2.to_dataframe()
cph2 = CoxPHFitter()
cph2.fit(df2, duration_col='T', event_col='E')
cph2.check_assumptions(df2, p_value_threshold=0.05, show_plots=True)

## Setup 3 — High-dimensional sparse
Expect: ~30-50% censoring, reasonable event time distribution.

In [ ]:
data3 = generate_setup_3()
print(data3.summary())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Event time histogram
axes[0].hist(data3.T[data3.E == 1], bins=50, alpha=0.7, label='Events')
axes[0].hist(data3.T[data3.E == 0], bins=50, alpha=0.7, label='Censored')
axes[0].set_title('Setup 3 — Observed Time Distribution')
axes[0].legend()

# KM curve
kmf3 = KaplanMeierFitter()
kmf3.fit(data3.T, event_observed=data3.E)
kmf3.plot_survival_function(ax=axes[1])
axes[1].set_title('Setup 3 — KM Curve')

plt.tight_layout()
plt.show()

## Setup 4 — Competing risks
Expect: ~30-40% censoring, two cause types, cumulative incidence <= 1.

In [ ]:
data4 = generate_setup_4()
print(data4.summary())

In [ ]:
# Cause-specific KM curves
fig, ax = plt.subplots(figsize=(8, 5))

for cause_val, label in [(1, 'Relapse'), (2, 'Direct Death')]:
    mask = (data4.cause == cause_val) | (data4.E == 0)
    E_cause = (data4.cause == cause_val).astype(int)
    kmf_c = KaplanMeierFitter()
    kmf_c.fit(data4.T, event_observed=E_cause, label=label)
    kmf_c.plot_survival_function(ax=ax)

ax.set_title('Setup 4 — Cause-Specific Survival')
plt.tight_layout()
plt.show()

# Transition counts
print('\nTransition counts (among events):')
events = data4.cause[data4.E == 1]
for c in [1, 2]:
    print(f'  Cause {c}: {(events == c).sum()}')

## Setup 5 — Large-n, heavy censoring
Expect: 60-70% censoring, fast generation.

In [ ]:
import time

t0 = time.perf_counter()
data5 = generate_setup_5()
elapsed = time.perf_counter() - t0

print(data5.summary())
print(f'Generation time: {elapsed:.2f}s')

In [ ]:
kmf5 = KaplanMeierFitter()
kmf5.fit(data5.T, event_observed=data5.E)

fig, ax = plt.subplots(figsize=(8, 5))
kmf5.plot_survival_function(ax=ax)
ax.set_title('Setup 5 — KM Curve (heavy censoring)')
plt.tight_layout()
plt.show()

## Covariate Correlation Checks

In [ ]:
# Setup 1 covariate correlation heatmap (should show AR(1) structure)
fig, ax = plt.subplots(figsize=(8, 6))
corr = np.corrcoef(data1.X.T)
sns.heatmap(corr, ax=ax, cmap='coolwarm', center=0,
            xticklabels=data1.feature_names, yticklabels=data1.feature_names)
ax.set_title('Setup 1 — Covariate Correlation')
plt.tight_layout()
plt.show()